# Simple: DateTime Generalization with PAMOLA.CORE

**Goal**: Learn datetime generalization basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply rounding-based datetime generalization using execute()
- Compare before/after results
- Understand privacy improvements through temporal precision reduction

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/generalization/
        └── 01_datetime_generalization_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.generalization.datetime_op import (
        DateTimeGeneralizationOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv

    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run this cell to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file doesn't exist
- Review dataset structure and datetime field format

**What you'll see:**
- File location confirmation and load status
- Dataset summary (rows, columns)
- First 5 records with all fields
- Column statistics showing:
  - Datetime fields: date range (min to max)
  - Numeric fields: min/max values
  - Categorical fields: unique value counts

**Data requirements:**
- Must contain at least one datetime field
- Datetime format: ISO 8601 or parseable format
- Sample data includes `snapshot_date` field (used in Step 3)

**Expected output:** DataFrame with datetime, categorical, and numeric columns

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with datetime patterns
    base_date = datetime(2024, 1, 1, 9, 0, 0)
    timestamps = []
    events = []
    locations = []
    
    for i in range(20):
        # Create varied timestamps
        offset_days = i * 3
        offset_hours = (i % 8) * 2
        offset_minutes = (i % 6) * 10
        
        ts = base_date + timedelta(
            days=offset_days, 
            hours=offset_hours, 
            minutes=offset_minutes
        )
        timestamps.append(ts)
        
        # Add event types
        event_types = ['Login', 'Purchase', 'View', 'Download', 'Logout']
        events.append(event_types[i % len(event_types)])
        
        # Add locations
        locs = ['Office', 'Home', 'Cafe', 'Airport', 'Hotel']
        locations.append(locs[i % len(locs)])
    
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'timestamp': timestamps,
        'event_type': events,
        'location': locations,
        'value': np.random.randint(10, 100, 20)
    })
    
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].min()} to {df[col].max()}")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE FIELD NAME**: Edit `create_config_kwargs()` function:
   - Change `"field_name": "snapshot_date"` to YOUR datetime column
   - For sample data: use `"snapshot_date"` (default)
   - For custom data: use your datetime column name
2. Run this cell to validate field and setup environment
3. Verify field validation passes (check output below)

**What you'll see:**
- Field validation results:
  - Field name confirmation
  - Unique value count
  - Date range (earliest to latest)
  - Sample datetime values (first 3)
- Environment setup confirmations:
  - Task directory created (✓)
  - TaskReporter initialized (✓)
  - DataSource created (✓)
  - Progress tracker ready (✓)

**⚠️ IMPORTANT:** 
- Field must exist in dataset (check Step 2 column list)
- Field must be datetime type (or parseable as datetime)
- If validation fails, verify field name matches exactly (case-sensitive)

**Configuration pattern:**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",  # Don't change
        "field_name": "snapshot_date"    # ← CUSTOMIZE THIS!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "snapshot_date"  # Field to generalize - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to generalize: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Date range: {df[field_name].min()} to {df[field_name].max()}")
print(f"  Sample values:")
for ts in df[field_name].head(3):
    print(f"    {ts}")

# Create task directory (in examples/data_examples)
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="anonymization",
    description=f"DateTime generalization of {field_name} field using rounding strategy.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration (production defaults set)
- Run to execute datetime generalization
- Monitor progress bar (6 steps: validation → parsing → rounding → metrics → viz → save)

**What you'll see:**
- Configuration summary (field, strategy, rounding unit, threshold)
- Progress bar with 6 processing steps
- Completion status = "completed" (verify no errors)
- Artifact count: 4-6 files expected

**Key parameters:**
- strategy='rounding': Rounds timestamps to reduce precision
- rounding_unit='day': Removes hours/minutes/seconds
- min_privacy_threshold=0.3: Requires ≥30% reduction
- mode='REPLACE': Overwrites original field

**Note:** Execution takes ~5-10 seconds. Rounding to 'day' means 2024-01-15 09:23:45 → 2024-01-15 00:00:00

In [ ]:
# Create operation with production-style configuration
operation = DateTimeGeneralizationOperation(
    # Core parameters
    field_name=field_name,
    mode='REPLACE',
    
    # Generalization strategy
    strategy='rounding',
    rounding_unit='day',  # Options: 'year', 'quarter', 'month', 'week', 'day', 'hour'
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Timezone handling
    timezone_handling='preserve',  # Options: 'preserve', 'utc', 'remove'
    default_timezone='UTC',
    
    # Privacy settings
    min_privacy_threshold=0.3,  # Minimum 30% reduction in unique values
    quasi_identifiers=None,
    
    # Null handling
    null_strategy='PRESERVE',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Strategy:         {operation.strategy}")
print(f"  Rounding unit:    {operation.rounding_unit}")
print(f"  Privacy threshold: {operation.min_privacy_threshold:.0%}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing datetime generalization...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare generalized data
- Review precision comparison (exact time → day only)
- Verify reduction meets privacy threshold (≥30%)

**What you'll see:**
- First 10 generalized records (times zeroed to 00:00:00)
- Before/after timestamp comparison (precision loss visible)
- Summary metrics:
  - Unique value reduction (e.g., 80 → 20 = 75%)
  - Date span preserved (range unchanged)
  - Privacy effectiveness ratio

**Validate:**
- ✅ Reduction ≥30% (meets threshold)
- ✅ All times = 00:00:00 (if unit='day')
- ✅ Date range unchanged
- ✅ No unexpected nulls

**Note:** Higher reduction = better privacy. Example: 100 → 25 unique dates = 4x privacy improvement

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    result_df[field_name] = pd.to_datetime(result_df[field_name])
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Generalized Records:")
    display_cols = [field_name]
    # Add optional columns if they exist
    for col in ['record_id', 'timestamp', 'event_type', 'location', 'value']:
        if col in result_df.columns:
            display_cols.append(col)
    print(result_df[display_cols].head(10))
    
    # Precision comparison
    print(f"\n📈 Temporal Precision Comparison:")
    print("\nOriginal data (first 5):")
    for ts in df[field_name].head(5):
        print(f"  {ts}")
    
    print("\nGeneralized data (first 5):")
    for ts in result_df[field_name].head(5):
        print(f"  {ts}")
    
    # Calculate metrics
    original_unique = df[field_name].nunique()
    generalized_unique = result_df[field_name].nunique()
    reduction_pct = (original_unique - generalized_unique) / original_unique * 100
    
    # Date range
    orig_span = (df[field_name].max() - df[field_name].min()).days
    gen_span = (result_df[field_name].max() - result_df[field_name].min()).days
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original unique values:    {original_unique}")
    print(f"  Generalized unique values: {generalized_unique}")
    print(f"  Reduction:                     {reduction_pct:.1f}%")
    print(f"  Original date span:            {orig_span} days")
    print(f"  Generalized date span:         {gen_span} days")
    
    # Display result metrics
    if result.metrics:
        print("\n🔒 Privacy Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Fewer unique values = Better privacy protection!")
    print(f"💡 Rounding to day removes exact timing information")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Generalized datetime CSV
├── metrics/          # Temporal precision & privacy metrics
├── visualizations/   # Before/after temporal distribution charts
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points in output

**What you'll see:**
1. **Metrics JSON**: Effectiveness ratio, privacy reduction %, datetime strategy, rounding unit
2. **Output Data**: First 10 rows with rounded timestamps, unique date count, sample dates
3. **Visualizations**: Temporal precision comparison, unique value reduction charts (latest batch only)

**Validate:**
- ✅ Effectiveness ratio ≥0.3 (threshold met)
- ✅ All times = 00:00:00 (rounding applied)
- ✅ Row count unchanged
- ✅ Visualizations show clear reduction

**Note:** Only newest files shown. Multiple test runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. Display Metrics JSON (NEWEST FILE)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types inside IF
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display effectiveness metrics
            if 'effectiveness' in metrics:
                print("\n📈 EFFECTIVENESS:")
                eff = metrics['effectiveness']
                print(f"   Total Records: {eff.get('total_records', 'N/A'):,}")
                print(f"   Original Unique: {eff.get('original_unique', 'N/A')}")
                print(f"   Anonymized Unique: {eff.get('anonymized_unique', 'N/A')}")
                print(f"   Effectiveness Ratio: {eff.get('effectiveness_ratio', 0):.4f}")
            
            # Display performance metrics
            print("\n⚡ PERFORMANCE:")
            print(f"   Duration: {metrics.get('duration_seconds', 0):.4f}s")
            print(f"   Records Processed: {metrics.get('records_processed', 0):,}")
            print(f"   Records/Second: {metrics.get('records_per_second', 0):.2f}")
            
            # Display datetime-specific metrics
            print("\n🕐 DATETIME-SPECIFIC METRICS:")
            datetime_keys = [
                ('datetime_strategy', 'Strategy'),
                ('rounding_unit', 'Rounding Unit'),
                ('unique_patterns_before', 'Patterns Before'),
                ('unique_patterns_after', 'Patterns After'),
                ('privacy_reduction_ratio', 'Privacy Reduction'),
                ('original_date_span_days', 'Date Span (days)')
            ]
            for key, label in datetime_keys:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float) and 0 < value < 1:
                        print(f"   {label}: {value:.2%}")
                    else:
                        print(f"   {label}: {value}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. Display Output Data Sample (NEWEST FILE)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            output_df[field_name] = pd.to_datetime(output_df[field_name])
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            if field_name in output_df.columns:
                print(f"\n\n📊 {field_name.title()} Unique Values:")
                print("-" * 80)
                unique_dates = output_df[field_name].unique()
                print(f"Total unique dates: {len(unique_dates)}")
                print("\nSample unique dates:")
                for date in sorted(unique_dates)[:5]:
                    print(f"  {pd.to_datetime(date)}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. Display Visualizations (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                name_parts = viz_file.stem.split('_')
                if len(name_parts) > 3:
                    viz_name = ' '.join(name_parts[2:-1]).strip().title()
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
            
            if len(viz_files) > len(latest_viz_batch):
                print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV with datetime fields from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with datetime generalization config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review temporal precision reduction and privacy metrics

**Key takeaways:**
- `operation.execute()` handles end-to-end processing
- Execution behavior (visualizations, output saving, caching) configured in operation constructor
- Field name is configurable via `create_config_kwargs()`
- Rounding strategy reduces temporal precision (e.g., exact time → day only)
- Fewer unique timestamps = better privacy protection
- Privacy threshold validates sufficient anonymization (30% reduction by default)
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_datetime_generalization_advanced.ipynb`** includes:
- All 4 strategies (rounding, binning, component, relative)
- Custom time bins and business hour categorization
- Seasonal and weekday patterns
- Timezone conversion and handling
- Advanced privacy metrics with k-anonymity
- Performance optimization for large datasets
- Production deployment patterns

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)